# ST7 Project 2026

## 0. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import glob
import os
import shutil
import sys
import time

import h5py
import numpy as np

# Add pysem source directory to sys.path if needed
path_to_src = str(Path("pysem/src").resolve())
if path_to_src not in sys.path:
    print(f"Adding {path_to_src} to sys.path")
    sys.path.append(path_to_src)

from pysem.parse_sem3d_traces import ParseSEM3DH5Traces

from util_funct.compute_misfit import compute_misfit
from util_funct.load_global_xyz_tuples import load_global_xyz_tuples
from util_funct.sbatch_and_wait import sbatch_and_wait
from util_funct.write_backward_spec import write_backward_spec_from_template
from util_funct.write_misfit_files import write_time_reversed_residual_files

from gradient_search_direction.util_funct.modify_h5 import modify_h5_g
from gradient_search_direction.util_funct.update_parameters_file import update_parameters_file

## 1. Paths

In [ ]:
# Base folders
SEM3D_CONFIG_RES_FOLDER_PATH = "./sem3d_config_files"
SEM3D_CONFIG_RES_FOLDER_PATH_ADJ = "./sem3d_config_files_adj"

# Forward problem
FORWARD_PROBLEM_MESHER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "MESHER.sbatch")
FORWARD_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "SOLVER.sbatch")

TRACES_SIMULATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "traces")
TRACES_OSSERVATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "Uobs")

# Adjoint problem
ADJOINT_SOURCES_FOLDER_PATH = SEM3D_CONFIG_RES_FOLDER_PATH_ADJ
ADJOINT_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "SOLVER_ADJOINT.sbatch")

STATIONS_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "stations.txt")

# Templates
BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "template/input_backward_template.spec")

In [ ]:
# Gradient / search direction
GRADIENT_SEARCH_DIRECTION_FOLDER_PATH = "./gradient_search_direction"

GRADIENT_SEARCH_DIRECTION_SBATCH_PATH = os.path.join(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH, "gradient_search_direction.sbatch")
GRADIENT_SEARCH_DIRECTION_PARAMETERS_SBATCH_PATH = os.path.join(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH, "gradient_search_parameters_file.json")

# L-BFGS
LBFGS_STATE_FOLDER_PATH = "state"
LBFGS_OUTPUT_FOLDER_PATH = "outputs"

## 2. Initial material

In [ ]:
# Generate material files
#! python3 ./pysem/src/pysem/generate_h5_materials.py @@prop "la" "mu" "ds" @@tag "linear_gradient" @@dir "z" @@xlim -1300 1300 @@ylim -1300 1300 @@zlim -1540 0 @@step 13 13 77 @@pfx 'example'
#! mv example* {SEM3D_CONFIG_RES_FOLDER_PATH} 
#! cp  {SEM3D_CONFIG_RES_FOLDER_PATH}/example* {SEM3D_CONFIG_RES_FOLDER_PATH_ADJ}

# Copy sample material files
! cp  ./materials_samples/example* {SEM3D_CONFIG_RES_FOLDER_PATH}
! cp  ./materials_samples/example* {SEM3D_CONFIG_RES_FOLDER_PATH_ADJ}

## 3. Clean old results

In [ ]:
# ============================================================
# CLEAN OUTPUT DIRECTORIES
# ============================================================

# Forward folders
! rm -r ./sem3d_config_files/mat
! rm -r ./sem3d_config_files/res
! rm -r ./sem3d_config_files/traces
! rm -r ./sem3d_config_files/mirror
! rm -r ./sem3d_config_files/prot
! rm -r ./sem3d_config_files/sem

# Adjoint folders
! rm -r ./sem3d_config_files_adj/mat
! rm -r ./sem3d_config_files_adj/res
! rm -r ./sem3d_config_files_adj/traces
! rm -r ./sem3d_config_files_adj/mirror
! rm -r ./sem3d_config_files_adj/prot
! rm -r ./sem3d_config_files_adj/sem

# Gradient / L-BFGS outputs
! rm -r ./gradient_search_direction/outputs/
! rm -r ./gradient_search_direction/state/
! rm ./gradient_search_direction/error.*
! rm ./gradient_search_direction/outputs/global_xyz_tuples.pkl
! rm ./gradient_search_direction/outputs/iter_*
! rm ./gradient_search_direction/mesh_mapping*
! rm ./gradient_search_direction/multi_hexahedral_mesh_*

## 4. Algorithm

In [ ]:
LBFGS_MEM = 5 # Number of iterations to remember
tol = 1e-4
n = 0 #Iter number
J = np.inf

In [ ]:
sbatch_and_wait(FORWARD_PROBLEM_MESHER_SBATCH_PATH)
! cp -r {SEM3D_CONFIG_RES_FOLDER_PATH}/sem {SEM3D_CONFIG_RES_FOLDER_PATH_ADJ}

stations = np.loadtxt(STATIONS_FILE_PATH)

while J >= tol:

    # ============================================================
    # STEP 1 — FORWARD PROBLEM
    # ============================================================

    sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)


    # ============================================================
    # STEP 2 — MISFIT COMPUTATION
    # ============================================================

    J, residual, t_sim, dt_sim = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, 
                                                TRACES_OSSERVATED_FOLDER_PATH)
      

    # ============================================================
    # STEP 3 — ADJOINT PROBLEM SETUP AND EXECUTION
    # ============================================================
    
    time_reversed_residual = residual[::-1, :, :]
    
    file_names = write_time_reversed_residual_files(time_reversed_residual, 
                                                  t_sim, 
                                                  OUTPUT_DIR=ADJOINT_SOURCES_FOLDER_PATH)

    write_backward_spec_from_template(template_backward_spec_path=BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH, 
                                      output_backward_spec_path = SEM3D_CONFIG_RES_FOLDER_PATH_ADJ,
                                      adjoint_sources_folder_path = ADJOINT_SOURCES_FOLDER_PATH, 
                                      stations = stations, 
                                      file_names = file_names)
    
    sbatch_and_wait(ADJOINT_PROBLEM_SOLVER_SBATCH_PATH)
    
    # ============================================================
    # STEP 4–5 — GRADIENT AND SEARCH DIRECTION COMPUTATION
    # ============================================================
    
    update_parameters_file(parameters_file_path=GRADIENT_SEARCH_DIRECTION_PARAMETERS_SBATCH_PATH, 
                           n_iter=n, 
                           LBFGS_MEM=LBFGS_MEM, 
                           LBFGS_STATE_FOLDER_PATH = LBFGS_STATE_FOLDER_PATH, 
                           LBFGS_OUTPUT_FOLDER_PATH = LBFGS_OUTPUT_FOLDER_PATH)
    
    sbatch_and_wait(GRADIENT_SEARCH_DIRECTION_SBATCH_PATH)


    # ============================================================
    # STEP 6 — BACKTRACKING LINE SEARCH
    # ============================================================
    
    # Safe loading of global vectors
    iter_folder = f"iter_{str(n).zfill(4)}"    
    vectors_file_path = (
        Path(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH).resolve()
        / LBFGS_OUTPUT_FOLDER_PATH
        / iter_folder
        / "global_vectors.npz"
    )

    max_tries = 30
    sleep_seconds = 2

    data = None
    last_err = None

    for _ in range(max_tries):
        if vectors_file_path.is_file():
            try:
                data = np.load(vectors_file_path)
                break
            except (OSError, EOFError, ValueError) as e:
                # file seen but maybe not fully visible / not fully flushed yet
                last_err = e
        time.sleep(sleep_seconds)

    if data is None:
        raise FileNotFoundError(
            f"Could not safely load file after waiting: {vectors_file_path}\n"
            f"Last error: {last_err}"
        )

    dir_la = data['dir_lam']
    dir_mu  = data['dir_mu']
    p1, p2 = data['scalar_prod_lam'], data['scalar_prod_mu']
    x_gl, y_gl, z_gl = data['x_gl'], data['y_gl'], data['z_gl']

    alpha_la=1 
    alpha_mu=1
    c1=1e-4 
    xi=0.5
    J_thresh = -np.inf
    J_learn = np.inf
        
    vec_to_add_la, vec_to_add_mu = 2*alpha_la*dir_la , 2*alpha_mu*dir_mu
    materials_paths = {'La': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH,"example_la.h5") , 
                       'Mu' : os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH,"example_mu.h5")} 
    mapping_list = [np.array([x_gl[i],y_gl[i],z_gl[i]]) for i in range(x_gl.size)]     


    modify_h5_g(materials_paths['La'],vec_to_add_la,mapping_list)
    modify_h5_g(materials_paths['Mu'],vec_to_add_mu,mapping_list)

    
    ls_iter = 0
    while J_learn >= J_thresh:
        print(f"--- Line Search Iteration {ls_iter} ---")        
        
        vec_to_add_la -= alpha_la*dir_la
        vec_to_add_mu -= alpha_mu*dir_mu

        modify_h5_g(materials_paths['La'],vec_to_add_la,mapping_list)
        modify_h5_g(materials_paths['Mu'],vec_to_add_mu,mapping_list)

        simulated_traces = glob.glob(os.path.join(TRACES_SIMULATED_FOLDER_PATH, "*"))
        for trace_file in simulated_traces:
            if os.path.isfile(trace_file):
                os.remove(trace_file)

        sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)

        J_learn, _, _, _ = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, TRACES_OSSERVATED_FOLDER_PATH)

        J_thresh = J + c1*(alpha_la*p1 + alpha_mu*p2)

        print(f"J_base   = {J:.15e}")
        print(f"J_learn  = {J_learn:.15e}")
        print(f"J_thresh = {J_thresh:.15e}")
        print(f"alpha_la = {alpha_la:.15e} | alpha_mu = {alpha_mu:.15e}")
        print(f"diff     = {(J_learn - J_thresh):.15e}")
        print("--------------------------------------------------")

        if J_learn < J_thresh:
            print("Armijo condition satisfied")
            break

        alpha_la *= xi
        alpha_mu *= xi
        
        ls_iter += 1    

    # ============================================================
    # STEP 7 — UPDATE ADJOINT MATERIAL FILES
    # ============================================================

    materials_paths_adj = {'La': os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ,"example_la.h5") , 
                           'Mu' : os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ,"example_mu.h5")}     
    vec_to_add_la_adj = -alpha_la/xi*vec_to_add_la
    vec_to_add_mu_adj = -alpha_mu/xi*vec_to_add_mu
    modify_h5_g(materials_paths_adj['La'],vec_to_add_la_adj,mapping_list)
    modify_h5_g(materials_paths_adj['Mu'],vec_to_add_mu_adj,mapping_list)

    n += 1    
